In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv("/Users/priyanshu.tuli/Desktop/machinehack/sequence_classification/Dataset/Train.csv")

In [3]:
train_df.head()

,Unnamed: 0,text,genre,label,label_model,text_cleaned
0,0,"It starts with pain, followed by hate\nFueled ...",rock,9,LABEL_9,"It starts with pain, followed by hate\nFueled ..."
1,1,Freedom!\nAlone again again alone\nPatiently w...,rock,9,LABEL_9,Freedom!\nAlone again again alone\nPatiently w...
2,2,"Biting the hand that feeds you, lying to the v...",rock,9,LABEL_9,"Biting the hand that feeds you, lying to the v..."
3,3,You say you know just who I am\nBut you can't ...,rock,9,LABEL_9,You say you know just who I am\nBut you can't ...
4,4,My heart is beating faster can't control these...,rock,9,LABEL_9,My heart is beating faster can't control these...


In [4]:
train_df.drop(columns=['Unnamed: 0'], inplace=True)

In [5]:
train_df.head()

,text,genre,label,label_model,text_cleaned
0,"It starts with pain, followed by hate\nFueled ...",rock,9,LABEL_9,"It starts with pain, followed by hate\nFueled ..."
1,Freedom!\nAlone again again alone\nPatiently w...,rock,9,LABEL_9,Freedom!\nAlone again again alone\nPatiently w...
2,"Biting the hand that feeds you, lying to the v...",rock,9,LABEL_9,"Biting the hand that feeds you, lying to the v..."
3,You say you know just who I am\nBut you can't ...,rock,9,LABEL_9,You say you know just who I am\nBut you can't ...
4,My heart is beating faster can't control these...,rock,9,LABEL_9,My heart is beating faster can't control these...


In [6]:
train_df.shape

(290183, 5)

In [7]:
from transformers import AutoTokenizer

/Users/priyanshu.tuli/Desktop/machinehack/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [40]:
def preprocess_function(examples):
    return tokenizer(examples['text_cleaned'], truncation=True)

In [11]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290183 entries, 0 to 290182
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   text          290148 non-null  object
 1   genre         290183 non-null  object
 2   label         290183 non-null  int64 
 3   label_model   290183 non-null  object
 4   text_cleaned  290147 non-null  object
dtypes: int64(1), object(4)
memory usage: 11.1+ MB


In [12]:
train_df.isna().sum()

text            35
genre            0
label            0
label_model      0
text_cleaned    36
dtype: int64

In [13]:
train_df.loc[train_df['text_cleaned'].isna(), ['label']]

,label
4080,7
4103,9
4114,9
4123,9
10491,9
22948,6
26925,9
27750,7
27921,7
41405,6


In [14]:
train_df.dropna(inplace=True)

In [16]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [17]:
from transformers import DataCollatorWithPadding

In [18]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [19]:
import evaluate

In [20]:
accuracy = evaluate.load("accuracy")

In [21]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [22]:
labels = train_df[['label', 'genre']].to_dict(orient='records')

In [23]:
labels = set([(label['label'], label['genre']) for label in labels])

In [24]:
labels

{(0, 'country'),
 (1, 'electronic'),
 (2, 'folk'),
 (3, 'hip-hop'),
 (4, 'indie'),
 (5, 'jazz'),
 (6, 'metal'),
 (7, 'pop'),
 (8, 'r&b'),
 (9, 'rock')}

In [25]:
id2label = {label[0]: label[1] for i, label in enumerate(labels)}
label2id = {label: i for i, label in id2label.items()}

In [26]:
id2label, label2id

({9: 'rock',
  8: 'r&b',
  5: 'jazz',
  6: 'metal',
  4: 'indie',
  0: 'country',
  7: 'pop',
  3: 'hip-hop',
  1: 'electronic',
  2: 'folk'},
 {'rock': 9,
  'r&b': 8,
  'jazz': 5,
  'metal': 6,
  'indie': 4,
  'country': 0,
  'pop': 7,
  'hip-hop': 3,
  'electronic': 1,
  'folk': 2})

In [27]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=10, id2label=id2label, label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
from sklearn.model_selection import train_test_split

In [33]:
train_df, eval_df = train_test_split(train_df, test_size=0.2)

In [37]:
req_train_df = train_df[['text_cleaned', 'label']]
req_eval_df = eval_df[['text_cleaned', 'label']]

In [47]:
req_train_dict = req_train_df.to_dict(orient='list')
req_eval_dict = req_eval_df.to_dict(orient='list')

In [48]:
req_eval_dict

{'text_cleaned': ["Pretty amazing grace is what You showed me\nPretty amazing grace is who You are\nI was an empty vessel\nYou filled me up inside\nAnd with amazing grace restored my pride\n\nPretty amazing grace is how You saved me\nAnd with amazing grace reclaimed my heart\nLove in the midst of chaos\nCalm in the heat of war\nShowed with amazing grace what love was for\n\nYou forgave my insensitivity\nAnd my attempt to then mislead You\nYou stood beside a wretch like me\nYour pretty amazing grace was all I needed.\n\nStumbled inside the doorway of Your chapel\nHumbled in God by everything I found\nBeauty and love surround me\nFreed me from what I fear\nAsk for amazing grace and You appear\n\nYou overcame my loss of hope and faith\nGave me a truth I could belive in\nYou led me to a higher place\nShowed Your amazing grace\nWhen grace was what I needed\n\nLook in a mirror I see Your reflection\nOpen a book You live on every page\nI fall and You're there to lift me\nShare every road I cl

In [ ]:
tokenized_train = preprocess_function(req_train_dict)
tokenized_eval = preprocess_function(req_eval_dict)

In [35]:
training_args = TrainingArguments(
    output_dir="./Dataset",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)


In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_text,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

ValueError: You have set `args.eval_strategy` to epoch but you didn't pass an `eval_dataset` to `Trainer`. Either set `args.eval_strategy` to `no` or pass an `eval_dataset`. 